<a href="https://colab.research.google.com/github/YasserJEP/TDA-BifurcacionesHopf/blob/main/Notebooks/7_Estimador_Par%C3%A1metro_Cr%C3%ADtico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este estimador identifica el parámetro crítico, mediante suavizado, en un conjunto de datos correspondientes a valores consecutivos del funcional topológico al variar el parámetro de control

# Hopf Normal

In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

# ============================================================
# CONFIGURACIÓN
# ============================================================
archivo_excel = r"D:\MAESTRÍA\TESIS\_MÁXIMA PERSISTENCIA\HOPF NORMAL\hopf_normal_max_persistence_mu_(-1.0,1.0)_N=300_Tau=26_m=2_noise=0.0.xlsx"  # <-- cambia esto
col_mu = "mu"
col_H = "max_persistence_H1"  # o como se llame tu columna de máxima persistencia

mu_teorico = 0.0   # valor teórico (Hopf normal)
sigma_suavizado = 2   # nivel de suavizado (1–3 recomendado)
ventana_refinamiento = 5  # tamaño ventana local

# ============================================================
# CARGA DE DATOS
# ============================================================
df = pd.read_excel(archivo_excel)

mu = df[col_mu].values
H = df[col_H].values

# Ordenar por mu (por seguridad)
idx = np.argsort(mu)
mu = mu[idx]
H = H[idx]

# ============================================================
# SUAVIZADO (ROBUSTEZ FRENTE A RUIDO)
# ============================================================
H_smooth = gaussian_filter1d(H, sigma=sigma_suavizado)

# ============================================================
# MÉTODO 1: DIFERENCIAS FINITAS (ΔH)
# ============================================================
delta_H = np.abs(np.diff(H_smooth))
j_star_diff = np.argmax(delta_H)
mu_c_diff = mu[j_star_diff + 1]

# ============================================================
# MÉTODO 2: DERIVADA DISCRETA (RECOMENDADO)
# ============================================================
dH = np.gradient(H_smooth, mu)
j_star_grad = np.argmax(np.abs(dH))
mu_c_grad = mu[j_star_grad]

# ============================================================
# REFINAMIENTO LOCAL (sobre derivada)
# ============================================================
start = max(0, j_star_grad - ventana_refinamiento)
end = min(len(mu) - 1, j_star_grad + ventana_refinamiento)

mu_local = mu[start:end]
H_local = H_smooth[start:end]

dH_local = np.gradient(H_local, mu_local)
j_local = np.argmax(np.abs(dH_local))

mu_c_refinado = mu_local[j_local]

# ============================================================
# ERRORES
# ============================================================
error_diff = abs(mu_c_diff - mu_teorico)
error_grad = abs(mu_c_grad - mu_teorico)
error_refinado = abs(mu_c_refinado - mu_teorico)

# ============================================================
# RESULTADOS EN CONSOLA
# ============================================================
print("\n===== RESULTADOS =====\n")

print(f"Estimación (ΔH): {mu_c_diff:.6f}")
print(f"Error (ΔH): {error_diff:.6e}\n")

print(f"Estimación (derivada): {mu_c_grad:.6f}")
print(f"Error (derivada): {error_grad:.6e}\n")

print(f"Estimación (refinada): {mu_c_refinado:.6f}")
print(f"Error (refinada): {error_refinado:.6e}")

# ============================================================
# TEXTO AUTOMÁTICO PARA LATEX
# ============================================================
latex_output = f"""
El parámetro crítico estimado mediante el criterio topológico es:

\\[
\\hat{{\\mu}}_c^{{\\mathrm{{TDA}}}} = {mu_c_refinado:.6f}
\\]

Comparado con el valor teórico $\\mu_c = {mu_teorico}$, se obtiene un error:

\\[
|\\hat{{\\mu}}_c^{{\\mathrm{{TDA}}}} - \\mu_c| = {error_refinado:.2e}
\\]
"""

print("\n===== LATEX =====\n")
print(latex_output)

# Lorenz

In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

# ============================================================
# CONFIGURACIÓN
# ============================================================
archivo_excel = r"D:\MAESTRÍA\TESIS\MÁXIMA PERSISTENCIA\LORENZ\Lorenz_max_persistence_rho_(20.0,30.0)_N=300_Tau=16_m=2_noise=0.0.xlsx" # <-- cambia esto

col_rho = "rho"
col_H = "max_persistence_H1"

rho_teorico = 24.74   # valor teórico de Lorenz
sigma_suavizado = 2
ventana_refinamiento = 5

# ============================================================
# CARGA DE DATOS
# ============================================================
df = pd.read_excel(archivo_excel)

rho = df[col_rho].values
H = df[col_H].values

# Ordenar por rho
idx = np.argsort(rho)
rho = rho[idx]
H = H[idx]

# ============================================================
# SUAVIZADO
# ============================================================
H_smooth = gaussian_filter1d(H, sigma=sigma_suavizado)

# ============================================================
# MÉTODO 1: DIFERENCIAS FINITAS (ΔH)
# ============================================================
delta_H = np.abs(np.diff(H_smooth))
j_star_diff = np.argmax(delta_H)
rho_c_diff = rho[j_star_diff + 1]

# ============================================================
# MÉTODO 2: DERIVADA DISCRETA
# ============================================================
dH = np.gradient(H_smooth, rho)
j_star_grad = np.argmax(np.abs(dH))
rho_c_grad = rho[j_star_grad]

# ============================================================
# REFINAMIENTO LOCAL
# ============================================================
start = max(0, j_star_grad - ventana_refinamiento)
end = min(len(rho), j_star_grad + ventana_refinamiento)

rho_local = rho[start:end]
H_local = H_smooth[start:end]

dH_local = np.gradient(H_local, rho_local)
j_local = np.argmax(np.abs(dH_local))

rho_c_refinado = rho_local[j_local]

# ============================================================
# ERRORES
# ============================================================
error_diff = abs(rho_c_diff - rho_teorico)
error_grad = abs(rho_c_grad - rho_teorico)
error_refinado = abs(rho_c_refinado - rho_teorico)

# ============================================================
# RESULTADOS
# ============================================================
print("\n===== RESULTADOS =====\n")

print(f"Estimación (ΔH): {rho_c_diff:.6f}")
print(f"Error (ΔH): {error_diff:.6e}\n")

print(f"Estimación (derivada): {rho_c_grad:.6f}")
print(f"Error (derivada): {error_grad:.6e}\n")

print(f"Estimación (refinada): {rho_c_refinado:.6f}")
print(f"Error (refinada): {error_refinado:.6e}")

# ============================================================
# TEXTO AUTOMÁTICO PARA LATEX
# ============================================================
latex_output = f"""
El parámetro crítico estimado mediante el criterio topológico es:

\\[
\\hat{{\\rho}}_c^{{\\mathrm{{TDA}}}} = {rho_c_refinado:.6f}
\\]

Comparado con el valor teórico $\\rho_c = {rho_teorico}$, se obtiene un error:

\\[
|\\hat{{\\rho}}_c^{{\\mathrm{{TDA}}}} - \\rho_c| = {error_refinado:.2e}
\\]
"""

print("\n===== LATEX =====\n")
print(latex_output)

# Reacción BZ

In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

# ============================================================
# CONFIGURACIÓN
# ============================================================
archivo_excel = r"D:\MAESTRÍA\TESIS\MÁXIMA PERSISTENCIA\REACCIÓN BZ\Reacción_BZ_max_persistence_b_(2.0,5.0)_N=300_Tau=58_m=2_noise=0.0_x02_y07.xlsx" # <-- cambia esto

col_b = "b"
col_H = "max_persistence_H1"

b_teorico = 3.5   # valor crítico teórico (BZ)
sigma_suavizado = 2
ventana_refinamiento = 5

# ============================================================
# CARGA DE DATOS
# ============================================================
df = pd.read_excel(archivo_excel)

b = df[col_b].values
H = df[col_H].values

# Ordenar por b
idx = np.argsort(b)
b = b[idx]
H = H[idx]

# ============================================================
# SUAVIZADO
# ============================================================
H_smooth = gaussian_filter1d(H, sigma=sigma_suavizado)

# ============================================================
# MÉTODO 1: DIFERENCIAS FINITAS (ΔH)
# ============================================================
delta_H = np.abs(np.diff(H_smooth))
j_star_diff = np.argmax(delta_H)
b_c_diff = b[j_star_diff + 1]

# ============================================================
# MÉTODO 2: DERIVADA DISCRETA
# ============================================================
dH = np.gradient(H_smooth, b)
j_star_grad = np.argmax(np.abs(dH))
b_c_grad = b[j_star_grad]

# ============================================================
# REFINAMIENTO LOCAL
# ============================================================
start = max(0, j_star_grad - ventana_refinamiento)
end = min(len(b), j_star_grad + ventana_refinamiento)

b_local = b[start:end]
H_local = H_smooth[start:end]

dH_local = np.gradient(H_local, b_local)
j_local = np.argmax(np.abs(dH_local))

b_c_refinado = b_local[j_local]

# ============================================================
# ERRORES
# ============================================================
error_diff = abs(b_c_diff - b_teorico)
error_grad = abs(b_c_grad - b_teorico)
error_refinado = abs(b_c_refinado - b_teorico)

# ============================================================
# RESULTADOS
# ============================================================
print("\n===== RESULTADOS =====\n")

print(f"Estimación (ΔH): {b_c_diff:.6f}")
print(f"Error (ΔH): {error_diff:.6e}\n")

print(f"Estimación (derivada): {b_c_grad:.6f}")
print(f"Error (derivada): {error_grad:.6e}\n")

print(f"Estimación (refinada): {b_c_refinado:.6f}")
print(f"Error (refinada): {error_refinado:.6e}")

# ============================================================
# TEXTO AUTOMÁTICO PARA LATEX
# ============================================================
latex_output = f"""
El parámetro crítico estimado mediante el criterio topológico es:

\\[
\\hat{{b}}_c^{{\\mathrm{{TDA}}}} = {b_c_refinado:.6f}
\\]

Comparado con el valor teórico $b_c = {b_teorico}$, se obtiene un error:

\\[
|\\hat{{b}}_c^{{\\mathrm{{TDA}}}} - b_c| = {error_refinado:.2e}
\\]
"""

print("\n===== LATEX =====\n")
print(latex_output)